# 07 — LLM Narration and BI Decision Support

**Research question**: can a generative LLM convert structured customer analytics into understandable, decision-oriented narratives while remaining grounded in the underlying analytical evidence — not "use Gemini to write pretty text."

```
Analytical models (Segment, Churn, CLV, SHAP, Forecast)
       |
Customer Insight (schema-validated structured context)
       |
Grounded prompt  ->  Gemini  ->  Structured narrative
       |
Faithfulness validation
```

The LLM is an **optional interpretation layer, not the source of truth** — if narration fails for a customer (API error, malformed output), the analytical results (segment/churn/CLV/SHAP/forecast) remain valid and available on their own; only the narrative is missing.

### Setup: Gemini API key
Add a Colab secret named `GEMINI_API_KEY` (key icon in the left sidebar) before running this notebook, and grant this notebook access to it when prompted.

In [12]:

from google.colab import drive
drive.mount('/content/drive')

!pip install -q duckdb pandas pyarrow scikit-learn xgboost lightgbm mlxtend shap prophet google-genai

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Ecommerce-AI-Business-Intelligence")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROCESSED_DIR / "features"
MODELS_DIR = PROCESSED_DIR / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
WAREHOUSE_PATH = PROJECT_ROOT / "warehouse.duckdb"

for d in [PROCESSED_DIR, FEATURES_DIR, MODELS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/Colab Notebooks/Ecommerce-AI-Business-Intelligence


In [13]:

from pathlib import Path
NARRATIVES_DIR = PROJECT_ROOT / "reports" / "narratives"
NARRATIVES_DIR.mkdir(parents=True, exist_ok=True)

from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
# Model name: adjust to whatever's current/available for your API key if this errors --
# Gemini model names change over time and this wasn't verified against a live key.
GEMINI_MODEL = "gemini-3.6-flash"

from google import genai
client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini client ready, model:", GEMINI_MODEL)


Gemini client ready, model: gemini-3.6-flash


## Load every analytical output

Everything produced by Notebooks 3-6.

In [14]:

import pandas as pd, json

customer_features = pd.read_parquet(FEATURES_DIR / "customer_features.parquet")
segments = pd.read_parquet(MODELS_DIR / "customer_segments.parquet")[["customer_unique_id", "segment_id"]]
segment_profile = pd.read_parquet(MODELS_DIR / "segment_profile.parquet")
churn_predictions = pd.read_parquet(MODELS_DIR / "churn_predictions.parquet")
clv_predictions = pd.read_parquet(MODELS_DIR / "clv_predictions.parquet")
shap_explanations = pd.read_parquet(MODELS_DIR / "shap_churn_customer_explanations.parquet")
with open(MODELS_DIR / "forecast_summary.json") as f:
    forecast_summary = json.load(f)

df = (customer_features
      .merge(segments, on="customer_unique_id", how="inner")
      .merge(churn_predictions[["customer_unique_id", "churn_probability"]], on="customer_unique_id", how="left")
      .merge(clv_predictions[["customer_unique_id", "clv_ml", "clv_formula"]], on="customer_unique_id", how="left")
      .merge(segment_profile[["segment_id", "segment_label"]], on="segment_id", how="left"))
print(f"{len(df):,} customers with full analytical context; {df['segment_id'].nunique()} segments")


95,420 customers with full analytical context; 4 segments


## Schema-validated customer insight

Structured context assembled *before* prompting, validated with Pydantic so a malformed/missing field fails loudly here rather than silently reaching the prompt.

In [15]:

from pydantic import BaseModel, Field
from typing import Optional

class SHAPDriver(BaseModel):
    feature: str
    feature_value: float
    shap_value: float
    direction: str

class CustomerInsight(BaseModel):
    customer_unique_id: str
    segment_id: int
    segment_label: Optional[str] = None
    frequency: int
    monetary: float
    avg_order_value: float
    recency_days: int
    churn_probability: float = Field(ge=0.0, le=1.0)
    clv_ml: float
    clv_formula: float
    shap_drivers: list[SHAPDriver]
    forecast_direction: str
    forecast_pct_change: float

def build_customer_insight(customer_unique_id: str) -> CustomerInsight:
    row = df[df["customer_unique_id"] == customer_unique_id].iloc[0]
    drivers = shap_explanations[shap_explanations["customer_unique_id"] == customer_unique_id]
    drivers = drivers.sort_values("shap_value", key=abs, ascending=False).head(5)
    return CustomerInsight(
        customer_unique_id=customer_unique_id,
        segment_id=int(row["segment_id"]),
        segment_label=row.get("segment_label"),
        frequency=int(row["frequency"]),
        monetary=float(row["monetary"]),
        avg_order_value=float(row["avg_order_value"]),
        recency_days=int(row["recency_days"]),
        churn_probability=float(row["churn_probability"]),
        clv_ml=float(row["clv_ml"]),
        clv_formula=float(row["clv_formula"]),
        shap_drivers=[SHAPDriver(**d) for d in drivers.to_dict("records")],
        forecast_direction=forecast_summary["direction"],
        forecast_pct_change=forecast_summary["pct_change"],
    )

# sanity check on one customer
example_insight = build_customer_insight(df["customer_unique_id"].iloc[0])
print(example_insight.model_dump_json(indent=2))


{
  "customer_unique_id": "0000366f3b9a7992bf8c76cfdf3221e2",
  "segment_id": 1,
  "segment_label": "Recent one-time buyers",
  "frequency": 1,
  "monetary": 141.9,
  "avg_order_value": 141.9,
  "recency_days": 117,
  "churn_probability": 0.25832444429397583,
  "clv_ml": 141.99966424977436,
  "clv_formula": 549.3092180728912,
  "shap_drivers": [
    {
      "feature": "avg_delivery_delay",
      "feature_value": -5.0,
      "shap_value": -0.8356320858001709,
      "direction": "decreases_churn_prediction"
    },
    {
      "feature": "avg_delivery_days",
      "feature_value": 6.0,
      "shap_value": -0.6046110987663269,
      "direction": "decreases_churn_prediction"
    },
    {
      "feature": "avg_freight",
      "feature_value": 12.0,
      "shap_value": 0.5427256226539612,
      "direction": "increases_churn_prediction"
    },
    {
      "feature": "avg_installments",
      "feature_value": 8.0,
      "shap_value": -0.04797683656215668,
      "direction": "decreases_churn_pre

## Grounded prompt and structured narrative schema

The model is instructed to reference **only** the supplied evidence and to use associative, not causal, language. Output is constrained to a fixed schema via `response_schema` — not free-form text — so it's machine-readable, comparable across customers, and easier to run the faithfulness check against.

In [16]:

class Narrative(BaseModel):
    summary: str = Field(description="2-3 sentence plain-language summary of this customer's situation")
    risk_explanation: str = Field(description="Why the model predicts this churn risk, grounded in the SHAP drivers")
    key_drivers: list[str] = Field(description="3-5 short phrases naming the strongest evidence, drawn only from "
                                                "the supplied SHAP feature names")
    recommended_actions: list[str] = Field(description="1-3 concrete, BI-oriented next actions")
    limitations: str = Field(description="What this analysis does NOT establish (e.g. causality)")

PROMPT_TEMPLATE = """You are a business-intelligence analyst writing a short, decision-oriented note about one e-commerce customer, for a human analyst who will act on it.

You must base every claim ONLY on the structured evidence below. Do not invent facts, feelings, or behaviors (e.g. "poor customer service experience") that are not present in this evidence. If the evidence is thin, say so in `limitations` rather than filling the gap with a plausible-sounding guess.

Use associative language, not causal language. Write "associated with a higher predicted churn risk", never "caused this customer to churn" -- the underlying models are predictive/associational, not causal.

CUSTOMER EVIDENCE (JSON):
{evidence}

Respond using the required structured schema only.
"""

def build_prompt(insight: CustomerInsight) -> str:
    return PROMPT_TEMPLATE.format(evidence=insight.model_dump_json(indent=2))


## Faithfulness checking (automated, lightweight)

Two heuristic checks, run after generation:
1. **Unsupported features** — does `key_drivers`/`risk_explanation` name a feature that isn't one of the features actually supplied in the SHAP evidence for that customer? (keyword match against the known feature-name vocabulary; not exhaustive NLP, but catches the clearest violations, e.g. an invented `customer_satisfaction` feature that was never in the context.)
2. **Causal language** — does the text use causal phrasing ("caused", "led to", "resulted in") instead of associative phrasing?

In [17]:

import re

KNOWN_FEATURE_VOCAB = set(shap_explanations["feature"].unique())
CAUSAL_PATTERNS = [
    r"\bcaused\b", r"\bcauses\b", r"\bcausing\b", r"\bled to\b", r"\bleads to\b",
    r"\bresulted in\b", r"\bresults in\b", r"\bdue to the fact that\b", r"\bbecause of this,?\s+the customer\b",
]

def check_faithfulness(narrative: Narrative, insight: CustomerInsight) -> dict:
    supplied_features = {d.feature for d in insight.shap_drivers}
    text = " ".join([narrative.summary, narrative.risk_explanation] + narrative.key_drivers).lower()

    mentioned_known_features = {f for f in KNOWN_FEATURE_VOCAB if f.lower().replace("_", " ") in text
                                 or f.lower() in text}
    unsupported_features = sorted(mentioned_known_features - supplied_features)

    causal_hits = [p for p in CAUSAL_PATTERNS if re.search(p, text, flags=re.IGNORECASE)]

    return {
        "unsupported_features": unsupported_features,
        "causal_language_flags": causal_hits,
        "passed": (len(unsupported_features) == 0) and (len(causal_hits) == 0),
    }


## Generate narration (graceful failure per customer)

In [18]:

def narrate_customer(customer_unique_id: str):
    """Returns (insight, narrative_or_None, faithfulness_or_None, error_or_None). Never raises --
    a failure here must not take down the surrounding analytics."""
    insight = build_customer_insight(customer_unique_id)
    try:
        response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=build_prompt(insight),
            config={"response_mime_type": "application/json", "response_schema": Narrative},
        )
        narrative = response.parsed
        faithfulness = check_faithfulness(narrative, insight)
        return insight, narrative, faithfulness, None
    except Exception as e:
        return insight, None, None, str(e)


## Evaluation sample: 7 customers x each segment, stratified by churn-risk tercile

Within each segment, customers are drawn across low/mid/high churn-probability terciles rather than uniformly at random, so the sample covers lower-, middle-, and higher-risk customers per segment.

In [19]:

import numpy as np

N_PER_SEGMENT = 7
SEED = 42
rng = np.random.default_rng(SEED)

sample_frames = []
for seg_id, seg_df in df.groupby("segment_id"):
    seg_df = seg_df.dropna(subset=["churn_probability"])
    if len(seg_df) < N_PER_SEGMENT:
        sample_frames.append(seg_df)
        continue
    tercile = pd.qcut(seg_df["churn_probability"], q=3, labels=["low", "mid", "high"], duplicates="drop")
    seg_df = seg_df.assign(risk_tercile=tercile)
    counts = [N_PER_SEGMENT // 3 + (1 if i < N_PER_SEGMENT % 3 else 0) for i in range(3)]
    picks = []
    for label, n in zip(["low", "mid", "high"], counts):
        pool = seg_df[seg_df["risk_tercile"] == label]
        n = min(n, len(pool))
        picks.append(pool.sample(n=n, random_state=SEED))
    sample_frames.append(pd.concat(picks))

eval_sample = pd.concat(sample_frames).reset_index(drop=True)
print(f"Evaluation sample: {len(eval_sample)} customers across {eval_sample['segment_id'].nunique()} segments")
eval_sample[["customer_unique_id", "segment_id", "churn_probability"]].head(10)


Evaluation sample: 28 customers across 4 segments


,customer_unique_id,segment_id,churn_probability
0,c63a14a83d3c4b14423a65d92d6e0a2f,0,0.481682
1,1b379c92f8e4520216fe60950c8007ff,0,0.512756
2,57dc5ffadb04793be6e216b39737061e,0,0.523483
3,c5c01882ea07c96a3647823205ae7187,0,0.627280
4,1a562770f891fc553db8662992cf1814,0,0.551417
5,c6d44715bc4825c73c205c39646c648a,0,0.704737
6,1a30d100192e3a5ad31164bf0dc90d9a,0,0.727419
7,2812b956488e289363a9dacfd67010f2,1,0.318945
8,874d4a75e09a5a708a37b529504f194a,1,0.114456
9,e49711ef161d2a621285a0ddd51a7d98,1,0.141034


In [20]:

results = []
for cust_id in eval_sample["customer_unique_id"]:
    insight, narrative, faithfulness, error = narrate_customer(cust_id)
    row = {"customer_unique_id": cust_id, "segment_id": insight.segment_id,
           "churn_probability": insight.churn_probability, "error": error}
    if narrative is not None:
        row.update({
            "summary": narrative.summary, "risk_explanation": narrative.risk_explanation,
            "key_drivers": " | ".join(narrative.key_drivers),
            "recommended_actions": " | ".join(narrative.recommended_actions),
            "limitations": narrative.limitations,
            "faithfulness_passed": faithfulness["passed"],
            "unsupported_features": ", ".join(faithfulness["unsupported_features"]),
            "causal_language_flags": ", ".join(faithfulness["causal_language_flags"]),
        })
    results.append(row)

narration_results = pd.DataFrame(results)
n_failed = narration_results["error"].notna().sum()
print(f"Narration generated for {len(narration_results) - n_failed}/{len(narration_results)} customers "
      f"({n_failed} failed -- analytics for those customers remain valid, only narration is missing)")


Narration generated for 19/28 customers (9 failed -- analytics for those customers remain valid, only narration is missing)


## Manual evaluation columns

Adds empty columns for the five human-scored dimensions, alongside the automated `faithfulness_passed` result, ready for manual annotation in Sheets/Excel.

In [21]:

for col in ["human_faithfulness_1to5", "human_numerical_accuracy_1to5", "human_clarity_1to5",
            "human_actionability_1to5", "human_non_causal_language_1to5", "human_notes"]:
    narration_results[col] = pd.NA

eval_path = NARRATIVES_DIR / "evaluation_sample.csv"
narration_results.to_csv(eval_path, index=False)
narration_results.to_parquet(NARRATIVES_DIR / "llm_narratives.parquet", index=False)
print("Saved", eval_path, "-- open in Sheets/Excel to score the five dimensions manually.")


Saved /content/drive/MyDrive/Colab Notebooks/Ecommerce-AI-Business-Intelligence/reports/narratives/evaluation_sample.csv -- open in Sheets/Excel to score the five dimensions manually.


## BI decision layer

Simple, transparent quadrant rules over churn x CLV, applied to the **full population** (not just the 28-customer sample). These are decision-support heuristics for prioritization, not experimentally validated causal interventions -- stated plainly so this isn't overclaimed in the paper.

In [22]:

CHURN_HIGH = 0.5
CLV_HIGH = df["clv_ml"].quantile(0.66)
RECENT_ONE_TIME_RECENCY_DAYS = df["recency_days"].median()

def decision_bucket(row):
    if row["frequency"] == 1 and row["churn_probability"] < CHURN_HIGH and row["recency_days"] <= RECENT_ONE_TIME_RECENCY_DAYS:
        return "recent_one_time_buyer", "Encourage conversion to a repeat customer"
    if row["churn_probability"] >= CHURN_HIGH and row["clv_ml"] >= CLV_HIGH:
        return "high_churn_high_clv", "Prioritize for retention efforts"
    if row["churn_probability"] >= CHURN_HIGH and row["clv_ml"] < CLV_HIGH:
        return "high_churn_low_clv", "Low-cost automated re-engagement rather than high-cost retention"
    if row["churn_probability"] < CHURN_HIGH and row["clv_ml"] >= CLV_HIGH:
        return "low_churn_high_clv", "Relationship maintenance, cross-sell / upsell"
    return "low_churn_low_clv", "No targeted intervention indicated"

bucket_results = df.apply(decision_bucket, axis=1, result_type="expand")
bucket_results.columns = ["decision_bucket", "suggested_action"]
decision_layer = pd.concat([df[["customer_unique_id", "segment_id", "churn_probability", "clv_ml"]],
                             bucket_results], axis=1)
decision_layer.to_parquet(MODELS_DIR / "decision_layer.parquet", index=False)

display(decision_layer["decision_bucket"].value_counts())


,count
decision_bucket,
high_churn_low_clv,33973
recent_one_time_buyer,32353
high_churn_high_clv,14391
low_churn_high_clv,7562
low_churn_low_clv,7141


---
### Outputs from this notebook
- `reports/narratives/evaluation_sample.csv` — 28 (7 x n_segments) customers, generated narratives, automated faithfulness flags, and blank columns for the 5 human-scored dimensions
- `reports/narratives/llm_narratives.parquet` — same data, parquet form
- `data/processed/models/decision_layer.parquet` — rule-based decision bucket for every customer

### A note on what to claim in the paper
- Report the **automated faithfulness results** as what they are: a keyword-based heuristic check, not a semantic/entailment verifier.
- Report **human-scored dimensions** only once `evaluation_sample.csv` has actually been scored.
- Present the decision layer as **decision-support rules**, explicitly not as validated causal interventions -- the quadrant thresholds (`CHURN_HIGH=0.5`, `CLV_HIGH`=66th percentile) are business choices, not statistically derived cutoffs.